In [13]:
import pandas as pd
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import pickle
from skimage.transform import rotate

In [ ]:
ROOT= './'

task1_path = os.path.join(ROOT, "subtask1")
task2_path = os.path.join(ROOT, "subtask2") 
task3_path = os.path.join(ROOT, "subtask3")


MODEL_PATH = os.path.join(ROOT, "doof_oracle.pkl")
with open(MODEL_PATH, "rb") as f:
    oracle = pickle.load(f)

result = []

from tqdm import tqdm
for path in tqdm(os.listdir(task1_path)):
    if not path.endswith('.npy'):
        continue
    IMAGE_PATH = os.path.join(task1_path, path)
    img = np.load(IMAGE_PATH)
    flat_img = img.reshape(1, -1)
    predicted_class = oracle.predict(flat_img)[0]

    if predicted_class == 0:
        pred = 0
    if predicted_class == 1:
        pred = 2
    if predicted_class == 2:
        rotated_img = rotate(img, angle=90, resize=False, order=1, preserve_range=True)
        flat_rotated_img = rotated_img.reshape(1, -1)
        predy = oracle.predict(flat_rotated_img)[0]
        if predy == 0:
            pred = 3
        else:
            pred = 1
    
    datapoint_id = int(path.split('.')[0])
    result.append({
        'subtaskID': 1,
        'datapointID': datapoint_id,
        'answer': pred
    })

for path in tqdm(os.listdir(task2_path)):
    if not path.endswith('.npy'):
        continue
    IMAGE_PATH = os.path.join(task2_path, path)
    img = np.load(IMAGE_PATH)
    flat_img = img.reshape(1, -1)
    predicted_class = oracle.predict(flat_img)[0] 

    probabilities = oracle.predict_proba(flat_img)[0]
    
    good_angle = 0
    for proba in probabilities:
        if proba > 0.95:
            good_angle = 1

    if good_angle == 1:
        if predicted_class == 0:
            pred = 0
        if predicted_class == 1:
            pred = 4
        if predicted_class == 2:
            rotated_img = rotate(img, angle=90, resize=False, order=1, preserve_range=True)
            flat_rotated_img = rotated_img.reshape(1, -1)
            predy = oracle.predict(flat_rotated_img)[0]
            if predy == 0:
                pred = 6
            else:
                pred = 2
    else:
        rotated_img = rotate(img, angle=45, resize=False, order=1, preserve_range=True)
        flat_rotated_img = rotated_img.reshape(1, -1)
        predy = oracle.predict(flat_rotated_img)[0]
        if predicted_class == 0:
            pred = 7
        if predicted_class == 1:
            pred = 3
        if predicted_class == 2:
            rotated_img = rotate(img, angle=90, resize=False, order=1, preserve_range=True)
            flat_rotated_img = rotated_img.reshape(1, -1)
            predy = oracle.predict(flat_rotated_img)[0]
            if predy == 0:
                pred = 1
            else:
                pred = 5
    
    datapoint_id = int(path.split('.')[0])
    result.append({
        'subtaskID': 2,
        'datapointID': datapoint_id,
        'answer': pred
    })

df_1 = pd.DataFrame(result)

ids = []
preds_0 = []
preds_1 = []
preds_2 = []
for path in tqdm(os.listdir(task3_path)):
    if not path.endswith('.npy'):
        continue
    IMAGE_PATH = os.path.join(task3_path, path)
    img = np.load(IMAGE_PATH)
    flat_img = img.reshape(1, -1)
    probabilities = oracle.predict_proba(flat_img)[0]
    preds_0.append(float(probabilities[0]))
    preds_1.append(float(probabilities[1]))
    preds_2.append(float(probabilities[2]))
    datapoint_id = int(path.split('.')[0])
    ids.append(datapoint_id)

 19%|██████████████▋                                                              | 951/5000 [00:00<00:03, 1099.83it/s]

In [15]:
df_outputs = pd.DataFrame({
    "preds0": preds_0,
    "preds1": preds_1,
    "preds2": preds_2
})

df_outputs.head()

,preds0,preds1,preds2
0,9.999996e-01,6.710130e-12,3.560145e-07
1,1.862062e-04,2.281466e-02,9.769991e-01
2,3.090809e-11,9.997957e-01,2.043327e-04
3,4.757595e-12,9.999963e-01,3.681188e-06
4,3.400743e-11,9.999803e-01,1.967754e-05


In [16]:
diffs = []
for _, row in df_outputs.iterrows():
    probs = [row.preds0, row.preds1, row.preds2]
    probs = sorted(probs, reverse=True)
    diff = probs[0] - probs[1]
    diffs.append(diff)

In [17]:
diffs = np.array(diffs)
threshold = 0.75
diffs = (diffs > threshold)
diffs

array([ True,  True,  True, ..., False, False, False])

In [18]:
df_2 = pd.DataFrame({
    'subtaskID': 3,
    'datapointID': ids,
    'answer': diffs
})

df_2.answer = df_2.answer.astype("int64")
df_2.answer = 1-df_2.answer

df_2.head()

,subtaskID,datapointID,answer
0,3,0,0
1,3,1,0
2,3,10,0
3,3,100,0
4,3,1000,0


In [19]:
df_2.answer.value_counts()

answer
0    4355
1     645
Name: count, dtype: int64

In [20]:
df_output = pd.concat([df_1, df_2])
df_output.head()

,subtaskID,datapointID,answer
0,1,0,1
1,1,1,0
2,1,10,1
3,1,100,0
4,1,1000,3


In [21]:
df_output[df_output.subtaskID == 2]

,subtaskID,datapointID,answer
5000,2,0,4
5001,2,1,6
5002,2,10,0
5003,2,100,6
5004,2,1000,0
...,...,...,...
9995,2,995,1
9996,2,996,3
9997,2,997,4
9998,2,998,3


In [22]:
df_output.to_csv('output.csv', index=False)